In [1]:
%pip install agent-framework python-dotenv 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import os
from dotenv import load_dotenv


load_dotenv()

True

In [ ]:
from agent_framework.openai import OpenAIChatClient


In [ ]:
#define local Ollama (OpenAI-compatible) router settings
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1")
CHAT_MODEL = os.getenv("OLLAMA_CHAT_MODEL", "llama3.1")

# Ollama ignores the key's value, but the OpenAI SDK requires a non-empty string
OLLAMA_API_KEY = os.getenv("OLLAMA_API_KEY", "ollama")


In [ ]:
client = OpenAIChatClient(
    base_url=OLLAMA_BASE_URL,
    api_key=OLLAMA_API_KEY,
    model=CHAT_MODEL,
)


In [15]:
#memory 

from dataclasses import dataclass, field, asdict
from typing import Optional


@dataclass
class ConversationMemory:
    """D2 — Conversation Memory: dates, occupancy, room type, meal plan."""

    check_in_date: Optional[str] = None
    check_out_date: Optional[str] = None
    adults_count: Optional[int] = None
    room_type: Optional[str] = None
    children_count: Optional[int] = None
    child_ages: list[int] = field(default_factory=list)
    meal_plan: Optional[str] = None

    def missing_fields(self) -> list[str]:
        missing = [
            field_name
            for field_name in (
                "check_in_date",
                "check_out_date",
                "adults_count",
                "room_type",
                "children_count",
                "meal_plan",
            )
            if getattr(self, field_name) is None
        ]

        if (
            self.children_count
            and self.children_count > 0
            and len(self.child_ages) < self.children_count
        ):
            missing.append("child_ages")

        return missing

    def is_complete(self) -> bool:
        return not self.missing_fields()

    def as_dict(self) -> dict:
        return asdict(self)


# One store per guest session.
# Same caveat as D1 in the Root Agent notebook.
d2_conversation_memory = ConversationMemory()


# Read-only mock Room Inventory — not part of D2.
# Supports the "show available room types" step (B4 in the Level 2 — P3 diagram).
ROOM_INVENTORY = {
    "Standard": {
        "available": 12,
        "max_adults": 2,
        "max_children": 1,
    },
    "Deluxe": {
        "available": 8,
        "max_adults": 2,
        "max_children": 2,
    },
    "Family Suite": {
        "available": 4,
        "max_adults": 3,
        "max_children": 3,
    },
    "Executive Suite": {
        "available": 3,
        "max_adults": 2,
        "max_children": 1,
    },
}


MEAL_PLAN_OPTIONS = [
    "Room Only",
    "Breakfast Included",
    "Half Board",
    "Full Board",
]

In [18]:
from datetime import datetime, date
from typing import Annotated, Optional

from agent_framework import tool


_DATE_FORMATS = (
    "%Y-%m-%d",
    "%d/%m/%Y",
    "%m/%d/%Y",
    "%d %B %Y",
    "%B %d, %Y",
    "%B %d %Y",
)


def _parse_date(value: str) -> Optional[date]:
    for fmt in _DATE_FORMATS:
        try:
            return datetime.strptime(value.strip(), fmt).date()
        except ValueError:
            continue

    return None


@tool
def record_check_in_date(
    check_in_date: Annotated[
        str,
        "The guest's requested check-in date, in any common format the guest used.",
    ],
) -> str:
    """Parse and save the check-in date to D2 (Conversation Memory)."""

    parsed = _parse_date(check_in_date)

    if not parsed:
        return (
            f"Couldn't understand '{check_in_date}' as a date. "
            "Ask the guest to re-enter it, e.g. YYYY-MM-DD."
        )

    d2_conversation_memory.check_in_date = parsed.isoformat()

    return f"Saved check-in date: {parsed.isoformat()}"


@tool
def record_check_out_date(
    check_out_date: Annotated[
        str,
        "The guest's requested check-out date, in any common format the guest used.",
    ],
) -> str:
    """Parse and save the check-out date to D2 (Conversation Memory). Must be after check-in."""

    parsed = _parse_date(check_out_date)

    if not parsed:
        return (
            f"Couldn't understand '{check_out_date}' as a date. "
            "Ask the guest to re-enter it, e.g. YYYY-MM-DD."
        )

    if d2_conversation_memory.check_in_date:
        check_in = date.fromisoformat(
            d2_conversation_memory.check_in_date
        )

        if parsed <= check_in:
            return (
                "Check-out date must be after the check-in date. "
                "Ask the guest to re-enter it."
            )

    d2_conversation_memory.check_out_date = parsed.isoformat()

    return f"Saved check-out date: {parsed.isoformat()}"


@tool
def record_adult_count(
    adults: Annotated[
        int,
        "How many adults the guest initially says will be staying.",
    ],
) -> str:
    """Save the guest's initial adult count to D2 (Conversation Memory)."""

    if adults < 1:
        return (
            "There must be at least one adult. "
            "Ask the guest to re-enter the count."
        )

    d2_conversation_memory.adults_count = adults

    return f"Saved initial adult count: {adults}"


@tool
def get_available_room_types() -> str:
    """
    Look up available room types and how many rooms are available in each,
    from Room Inventory.

    Does not write to D2 — it is a read-only lookup to show the guest options.
    """

    lines = [
        (
            f"{name}: {info['available']} available "
            f"(max {info['max_adults']} adults, "
            f"{info['max_children']} children)"
        )
        for name, info in ROOM_INVENTORY.items()
        if info["available"] > 0
    ]

    if not lines:
        return "No room types currently have availability."

    return "\n".join(lines)


@tool
def record_selected_room_type(
    room_type: Annotated[
        str,
        "The room type name the guest selected, exactly as shown to them.",
    ],
) -> str:
    """Save the guest's selected room type to D2 after checking availability."""

    info = ROOM_INVENTORY.get(room_type)

    if not info:
        return (
            f"'{room_type}' isn't a recognized room type. "
            "Show the available types again."
        )

    if info["available"] <= 0:
        return (
            f"'{room_type}' has no rooms available. "
            "Ask the guest to pick a different type."
        )

    d2_conversation_memory.room_type = room_type

    return f"Saved selected room type: {room_type}"


@tool
def record_occupancy(
    adults: Annotated[
        int,
        "Final number of adults for the booking, step 1 of the 2-step occupancy question.",
    ],
    children: Annotated[
        int,
        "Number of children for the booking, step 2 of the 2-step occupancy question.",
    ],
) -> str:
    """
    Save the final adult and children counts to D2, capped by the
    max occupancy of the previously selected room type.

    Call record_selected_room_type first.
    """

    if not d2_conversation_memory.room_type:
        return (
            "No room type has been selected yet. "
            "Ask the guest to pick a room type first."
        )

    limits = ROOM_INVENTORY[
        d2_conversation_memory.room_type
    ]

    if adults < 1 or adults > limits["max_adults"]:
        return (
            f"{d2_conversation_memory.room_type} allows at most "
            f"{limits['max_adults']} adults. Ask again."
        )

    if children < 0 or children > limits["max_children"]:
        return (
            f"{d2_conversation_memory.room_type} allows at most "
            f"{limits['max_children']} children. Ask again."
        )

    d2_conversation_memory.adults_count = adults
    d2_conversation_memory.children_count = children

    # Reset ages; record_child_ages fills this if children > 0.
    d2_conversation_memory.child_ages = []

    return f"Saved occupancy: {adults} adults, {children} children"


@tool
def record_child_ages(
    ages: Annotated[
        list[int],
        "One age per child, in the order the guest gave them.",
    ],
) -> str:
    """Save the children's ages to D2. Only call this if children_count > 0."""

    expected = d2_conversation_memory.children_count

    if not expected:
        return (
            "No children were recorded for this booking, "
            "so ages aren't needed."
        )

    if len(ages) != expected:
        return (
            f"Expected {expected} age(s) but got {len(ages)}. "
            "Ask the guest again."
        )

    d2_conversation_memory.child_ages = list(ages)

    return f"Saved child ages: {ages}"


@tool
def record_meal_plan(
    meal_plan: Annotated[
        str,
        "The meal plan the guest selected, exactly as shown to them.",
    ],
) -> str:
    """Save the guest's selected meal plan to D2 (Conversation Memory)."""

    match = next(
        (
            option
            for option in MEAL_PLAN_OPTIONS
            if option.lower() == meal_plan.strip().lower()
        ),
        None,
    )

    if not match:
        return (
            f"'{meal_plan}' isn't one of the meal plan options: "
            f"{', '.join(MEAL_PLAN_OPTIONS)}."
        )

    d2_conversation_memory.meal_plan = match

    return f"Saved meal plan: {match}"


@tool
def check_reservation_status() -> str:
    """Check which reservation fields are still missing from D2."""

    missing = d2_conversation_memory.missing_fields()

    if not missing:
        return "All reservation fields are recorded."

    return f"Still missing: {', '.join(missing)}"


@tool
def route_to_booking_policy_agent() -> str:
    """
    Route to the Booking Policy Agent (P4) when the guest has
    a policy question.

    P4 is not implemented in this notebook, so this is a stub
    that confirms the handoff.
    """

    return (
        "Handoff to Booking Policy Agent (P4) complete "
        "(stub — P4 isn't implemented in this notebook). "
        "Control would normally return here once the question is answered."
    )


@tool
def return_to_coordinator_agent() -> str:
    """
    Return control to the Coordinator Agent (P2) once the
    reservation is complete.

    P2 is not implemented in this notebook, so this is a stub
    that returns the D2 payload P2 would receive.
    """

    if not d2_conversation_memory.is_complete():
        missing = ", ".join(
            d2_conversation_memory.missing_fields()
        )

        return (
            "Cannot return to the Coordinator Agent yet — "
            f"still missing: {missing}."
        )

    return (
        "Returned to Coordinator Agent (P2). "
        f"Reservation payload: {d2_conversation_memory.as_dict()}"
    )


reservation_agent_tools = [
    record_check_in_date,
    record_check_out_date,
    record_adult_count,
    get_available_room_types,
    record_selected_room_type,
    record_occupancy,
    record_child_ages,
    record_meal_plan,
    check_reservation_status,
    route_to_booking_policy_agent,
    return_to_coordinator_agent,
]

In [16]:
RESERVATION_AGENT_INSTRUCTIONS = """
You are the Reservation Agent for a hotel booking assistant. The guest has already
been identified by the Root Agent. Your job is to collect every detail needed for
their room reservation, one step at a time, then hand back to the Coordinator Agent.

Do not answer policy, pricing, or payment questions yourself — route policy questions
to the Booking Policy Agent, and leave pricing/payment to the agents that own them.

Follow this exact sequence, asking only one question per turn:

1. Ask for the check-in date. Once given, call record_check_in_date.

2. Ask for the check-out date. Once given, call record_check_out_date. If the tool
   reports an issue (unparseable, or not after check-in), ask again.

3. Ask how many adults will be staying. Once given, call record_adult_count.

4. Call get_available_room_types and show the guest the room types, availability
   counts, and max occupancy per type.

5. Ask the guest to pick one of the available room types. Once chosen, call
   record_selected_room_type. If it's invalid or unavailable, show the list again.

6. Ask for the final occupancy in two steps: first "how many adults", then "how many
   children" — capped by the selected room type's max occupancy. Once you have both,
   call record_occupancy. If the tool reports the counts exceed the room type's
   limits, ask again.

7. If children_count > 0, ask for each child's age, then call record_child_ages with
   all of them together. Skip this step entirely if there are no children.

8. Ask the guest to choose a meal plan from: Room Only, Breakfast Included, Half
   Board, Full Board. Once chosen, call record_meal_plan.

9. Ask: "Do you have any booking-policy related questions?" If yes, call
   route_to_booking_policy_agent, relay its response, then continue. If no, continue.

10. Call check_reservation_status to confirm everything is recorded, then call
    return_to_coordinator_agent, and tell the guest in one short, friendly sentence
    that their reservation details are set and you're handing them back for the next
    step.

Rules:

- Be concise and friendly. Ask for exactly one piece of information at a time.
- Never fabricate or guess a value; only save what the guest actually provided.
- Never call return_to_coordinator_agent until check_reservation_status confirms
  nothing is missing.
- Do not re-ask for a field that has already been recorded successfully.
""".strip()

In [ ]:
from agent_framework import Agent

reservation_agent = Agent(
    client=client,
    name="ReservationAgent",
    description="Collects dates, occupancy, room selection, and meal plan (P3 in the hotel booking DFD).",
    instructions=RESERVATION_AGENT_INSTRUCTIONS,
    tools=reservation_agent_tools,
)

In [23]:
import asyncio


async def run_scripted_demo() -> None:
    session = reservation_agent.create_session()

    guest_turns = [
        "I'd like to book a room.",
        "Check-in on 2026-12-10.",
        "Check-out on 2026-12-14.",
        "3 adults to start with.",
        "We'll take the Family Suite.",
        "2 adults and 1 child.",
        "The child is 8 years old.",
        "Breakfast Included, please.",
        "No, no policy questions.",
    ]

    for turn in guest_turns:
        print(f"Guest: {turn}")
        result = await reservation_agent.run(turn, session=session)
        print(f"ReservationAgent: {result.text}\n")

    print("--- D2 (Conversation Memory) after the conversation ---")
    print(d2_conversation_memory.as_dict())


await run_scripted_demo()

Guest: I'd like to book a room.
ReservationAgent: Welcome! I'd be happy to help you with your booking. What date would you like to check in?

Guest: Check-in on 2026-12-10.
ReservationAgent: Thank you! What date will you be checking out?

Guest: Check-out on 2026-12-14.


ChatClientException: ("Gemini chat request failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}", ServerError("503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}"))